In [1]:
# Install required libraries for profiling
!pip install thop -q
!pip install codecarbon -q
!pip install scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 23.3 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 64.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed

# **CNN {Base Model}**

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time
import os
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from thop import profile
from codecarbon import track_emissions

# --- Configuration ---
BATCH_SIZE = 128  
EPOCHS = 100
LEARNING_RATE = 0.001
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"--- Base CNN on CIFAR-10 ---")
print(f"Using device: {DEVICE}")

 
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class BaseCNN(nn.Module):
    """A standard CNN adapted for CIFAR-10."""
    def __init__(self):
        super(BaseCNN, self).__init__()
        # Input: 32x32x3 image
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        # After pool1: 16x16x16
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        # After pool2: 8x8x32
        
        self.fc1 = nn.Linear(32 * 8 * 8, 128) # Adapted for 32x32 input
        self.fc2 = nn.Linear(128, 10) # 10 classes for CIFAR-10
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 32 * 8 * 8) # Flatten
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = BaseCNN().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

--- Base CNN on CIFAR-10 ---
Using device: cuda


100%|██████████| 170M/170M [00:05<00:00, 33.5MB/s] 


In [3]:
def train_model(model, device, train_loader, optimizer, criterion, epochs):
    model.train()
    print("--- Starting Training ---")
    for epoch in range(epochs):
        epoch_loss = 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        if (epoch + 1) % 10 == 0: # Print progress every 10 epochs
            print(f"Epoch {epoch+1}/{epochs}, Average Loss: {epoch_loss/len(train_loader):.4f}")
    print("--- Training Finished ---")

@track_emissions(project_name="BaseCNN_CIFAR10_Evaluation")
def evaluate_model(model, device, test_loader):
    model.eval()
    all_preds, all_targets = [], []
    total_latency, num_samples = 0, 0
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            start_time = time.perf_counter()
            output = model(data)
            total_latency += (time.perf_counter() - start_time)
            num_samples += data.size(0)
            
            _, predicted = torch.max(output.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
            
    return all_targets, all_preds, (total_latency / num_samples) * 1000


# --- Train and Evaluate ---
train_model(model, DEVICE, train_loader, optimizer, criterion, epochs=EPOCHS)
true_labels, predicted_labels, avg_latency_ms = evaluate_model(model, DEVICE, test_loader)

# --- Performance Metrics ---
print("\n--- Performance Metrics (Base CNN on CIFAR-10) ---")
accuracy = accuracy_score(true_labels, predicted_labels)
precision, recall, f1_score, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='macro', zero_division=0)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Precision (Macro): {precision:.4f}")
print(f"Recall (Macro): {recall:.4f}")
print(f"F1-Score (Macro): {f1_score:.4f}")

# --- Green AI / Efficiency Metrics ---
print("\n--- Green AI / Efficiency Metrics (Base CNN on CIFAR-10) ---")

# 1. Parameter Count & Model Size
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameter Count: {param_count / 1e6:.3f} M")
torch.save(model.state_dict(), "base_cnn_cifar10.pth")
model_size_mb = os.path.getsize("base_cnn_cifar10.pth") / (1024 * 1024)
os.remove("base_cnn_cifar10.pth")
print(f"Model Size: {model_size_mb:.3f} MB")

# 2. Computational Complexity (FLOPs)
dummy_input = torch.randn(1, 3, 32, 32).to(DEVICE)
flops, _ = profile(model, inputs=(dummy_input,), verbose=False)
print(f"Computational Complexity: {flops / 1e9:.4f} GFLOPs")

# 3. Inference Latency
print(f"Average Inference Latency: {avg_latency_ms:.4f} ms/image")

# 4. Energy Consumption
print("\nEnergy/CO2 metrics for the evaluation phase tracked by CodeCarbon.")
print("Check the 'emissions.csv' file in the output directory for details.")

--- Starting Training ---
Epoch 10/100, Average Loss: 0.8911
Epoch 20/100, Average Loss: 0.7700
Epoch 30/100, Average Loss: 0.7057
Epoch 40/100, Average Loss: 0.6683
Epoch 50/100, Average Loss: 0.6486
Epoch 60/100, Average Loss: 0.6300
Epoch 70/100, Average Loss: 0.6179
Epoch 80/100, Average Loss: 0.6025
Epoch 90/100, Average Loss: 0.5926


[codecarbon WARNING @ 12:04:44] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 12:04:44] [setup] RAM Tracking...
[codecarbon INFO @ 12:04:44] [setup] CPU Tracking...


Epoch 100/100, Average Loss: 0.5844
--- Training Finished ---


[codecarbon WARNING @ 12:04:45] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 12:04:45] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 12:04:45] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.00GHz
[codecarbon WARNING @ 12:04:45] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 12:04:45] [setup] GPU Tracking...
[codecarbon INFO @ 12:04:45] Tracking Nvidia GPU via pynvml
[codecarbon WARNING @ 12:04:45] Failed to retrieve gpu total energy consumption
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/codecarbon/core/gpu.py", line 116, in _get_total_energy_consumption
    return pynvml.nvmlDeviceGetTotalEnergyConsumption(self.handle)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


--- Performance Metrics (Base CNN on CIFAR-10) ---
Accuracy: 77.80%
Precision (Macro): 0.7814
Recall (Macro): 0.7780
F1-Score (Macro): 0.7776

--- Green AI / Efficiency Metrics (Base CNN on CIFAR-10) ---
Parameter Count: 0.269 M
Model Size: 1.028 MB
Computational Complexity: 0.0019 GFLOPs
Average Inference Latency: 0.0067 ms/image

Energy/CO2 metrics for the evaluation phase tracked by CodeCarbon.
Check the 'emissions.csv' file in the output directory for details.


#  **EfficientNet-B0**

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import time
import os
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from thop import profile
from codecarbon import track_emissions

# --- Configuration ---
BATCH_SIZE = 128
EPOCHS = 100
LEARNING_RATE = 0.001
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"--- EfficientNet-B0 on CIFAR-10 ---")
print(f"Using device: {DEVICE}")

# --- Data Loading & Transformation for CIFAR-10 ---
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# --- Load and Adapt EfficientNet-B0 for CIFAR-10 ---
model = models.efficientnet_b0(weights=None) # Train from scratch

# Adapt the final classifier layer for 10 CIFAR-10 classes
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 10)

model = model.to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()




def train_model(model, device, train_loader, optimizer, criterion, epochs):
    model.train()
    print("--- Starting Training ---")
    for epoch in range(epochs):
        epoch_loss = 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        if (epoch + 1) % 10 == 0: # Print progress every 10 epochs
            print(f"Epoch {epoch+1}/{epochs}, Average Loss: {epoch_loss/len(train_loader):.4f}")
    print("--- Training Finished ---")

@track_emissions(project_name="BaseCNN_CIFAR10_Evaluation")
def evaluate_model(model, device, test_loader):
    model.eval()
    all_preds, all_targets = [], []
    total_latency, num_samples = 0, 0
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            start_time = time.perf_counter()
            output = model(data)
            total_latency += (time.perf_counter() - start_time)
            num_samples += data.size(0)
            
            _, predicted = torch.max(output.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
            
    return all_targets, all_preds, (total_latency / num_samples) * 1000


# --- Train and Evaluate ---
train_model(model, DEVICE, train_loader, optimizer, criterion, epochs=EPOCHS)
true_labels, predicted_labels, avg_latency_ms = evaluate_model(model, DEVICE, test_loader)

# --- Performance Metrics ---
print("\n--- Performance Metrics (EfficientNet-B0 on CIFAR-10) ---")
accuracy = accuracy_score(true_labels, predicted_labels)
precision, recall, f1_score, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='macro', zero_division=0)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Precision (Macro): {precision:.4f}")
print(f"Recall (Macro): {recall:.4f}")
print(f"F1-Score (Macro): {f1_score:.4f}")

# --- Green AI / Efficiency Metrics ---
print("\n--- Green AI / Efficiency Metrics (EfficientNet-B0 on CIFAR-10) ---")

# 1. Parameter Count & Model Size
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameter Count: {param_count / 1e6:.3f} M")
torch.save(model.state_dict(), "efficientnet_b0_cifar10.pth")
model_size_mb = os.path.getsize("efficientnet_b0_cifar10.pth") / (1024 * 1024)
os.remove("efficientnet_b0_cifar10.pth")
print(f"Model Size: {model_size_mb:.3f} MB")

# 2. Computational Complexity (FLOPs)
dummy_input = torch.randn(1, 3, 32, 32).to(DEVICE)
flops, _ = profile(model, inputs=(dummy_input,), verbose=False)
print(f"Computational Complexity: {flops / 1e9:.4f} GFLOPs")

# 3. Inference Latency
print(f"Average Inference Latency: {avg_latency_ms:.4f} ms/image")

# 4. Energy Consumption
print("\nEnergy/CO2 metrics for the evaluation phase tracked by CodeCarbon.")
print("Check the 'emissions.csv' file for details.")

--- EfficientNet-B0 on CIFAR-10 ---
Using device: cuda
--- Starting Training ---
Epoch 10/100, Average Loss: 0.9336
Epoch 20/100, Average Loss: 0.7527
Epoch 30/100, Average Loss: 0.6537
Epoch 40/100, Average Loss: 0.6317
Epoch 50/100, Average Loss: 0.4919
Epoch 60/100, Average Loss: 0.3995
Epoch 70/100, Average Loss: 0.3480
Epoch 80/100, Average Loss: 0.3168
Epoch 90/100, Average Loss: 0.2885


[codecarbon WARNING @ 12:42:51] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 12:42:51] [setup] RAM Tracking...
[codecarbon INFO @ 12:42:51] [setup] CPU Tracking...


Epoch 100/100, Average Loss: 0.2277
--- Training Finished ---


[codecarbon WARNING @ 12:42:52] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 12:42:52] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 12:42:52] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.00GHz
[codecarbon WARNING @ 12:42:52] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 12:42:52] [setup] GPU Tracking...
[codecarbon INFO @ 12:42:52] Tracking Nvidia GPU via pynvml
[codecarbon WARNING @ 12:42:52] Failed to retrieve gpu total energy consumption
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/codecarbon/core/gpu.py", line 116, in _get_total_energy_consumption
    return pynvml.nvmlDeviceGetTotalEnergyConsumption(self.handle)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


--- Performance Metrics (EfficientNet-B0 on CIFAR-10) ---
Accuracy: 83.94%
Precision (Macro): 0.8398
Recall (Macro): 0.8394
F1-Score (Macro): 0.8388

--- Green AI / Efficiency Metrics (EfficientNet-B0 on CIFAR-10) ---
Parameter Count: 4.020 M
Model Size: 15.626 MB
Computational Complexity: 0.0091 GFLOPs
Average Inference Latency: 0.1237 ms/image

Energy/CO2 metrics for the evaluation phase tracked by CodeCarbon.
Check the 'emissions.csv' file for details.


# **ResNet-18**

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import time
import os
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from thop import profile
from codecarbon import track_emissions

# --- Configuration ---
BATCH_SIZE = 128
EPOCHS = 100
LEARNING_RATE = 0.001
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"--- ResNet-18 on CIFAR-10 ---")
print(f"Using device: {DEVICE}")

# --- Data Loading & Transformation for CIFAR-10 ---
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# --- Load and Adapt ResNet-18 for CIFAR-10 ---
model = models.resnet18(weights=None) # Train from scratch
 
model.conv1 = nn.Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
model.maxpool = nn.Identity() # We also remove the initial max pooling layer.

# Adapt the final fully connected layer for 10 CIFAR-10 classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)

model = model.to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()



# --- Training and Evaluation Functions (re-usable) ---
def train_model(model, device, train_loader, optimizer, criterion, epochs):
    model.train()
    print("--- Starting Training ---")
    for epoch in range(epochs):
        epoch_loss = 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad(); output = model(data); loss = criterion(output, target)
            loss.backward(); optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} complete.")
    print("--- Training Finished ---")

@track_emissions(project_name="ResNet18_CIFAR10_Evaluation")
def evaluate_model(model, device, test_loader):
    model.eval()
    all_preds, all_targets, total_latency, num_samples = [], [], 0, 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            start_time = time.perf_counter()
            output = model(data)
            total_latency += (time.perf_counter() - start_time)
            num_samples += data.size(0)
            _, predicted = torch.max(output.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
    return all_targets, all_preds, (total_latency / num_samples) * 1000

 
# --- Train and Evaluate ---
train_model(model, DEVICE, train_loader, optimizer, criterion, epochs=EPOCHS)
true_labels, predicted_labels, avg_latency_ms = evaluate_model(model, DEVICE, test_loader)

# --- Performance Metrics ---
print("\n--- Performance Metrics (ResNet-18 on CIFAR-10) ---")
accuracy = accuracy_score(true_labels, predicted_labels)
precision, recall, f1_score, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='macro', zero_division=0)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Precision (Macro): {precision:.4f}")
print(f"Recall (Macro): {recall:.4f}")
print(f"F1-Score (Macro): {f1_score:.4f}")

# --- Green AI / Efficiency Metrics ---
print("\n--- Green AI / Efficiency Metrics (ResNet-18 on CIFAR-10) ---")

# 1. Parameter Count & Model Size
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameter Count: {param_count / 1e6:.3f} M")
torch.save(model.state_dict(), "resnet18_cifar10.pth")
model_size_mb = os.path.getsize("resnet18_cifar10.pth") / (1024 * 1024)
os.remove("resnet18_cifar10.pth")
print(f"Model Size: {model_size_mb:.3f} MB")

# 2. Computational Complexity (FLOPs)
dummy_input = torch.randn(1, 3, 32, 32).to(DEVICE)
flops, _ = profile(model, inputs=(dummy_input,), verbose=False)
print(f"Computational Complexity: {flops / 1e9:.4f} GFLOPs")

# 3. Inference Latency
print(f"Average Inference Latency: {avg_latency_ms:.4f} ms/image")

# 4. Energy Consumption
print("\nEnergy/CO2 metrics for the evaluation phase tracked by CodeCarbon.")
print("Check the 'emissions.csv' file for details.")

--- ResNet-18 on CIFAR-10 ---
Using device: cuda
--- Starting Training ---
Epoch 10/100 complete.
Epoch 20/100 complete.
Epoch 30/100 complete.
Epoch 40/100 complete.
Epoch 50/100 complete.
Epoch 60/100 complete.
Epoch 70/100 complete.
Epoch 80/100 complete.
Epoch 90/100 complete.


[codecarbon WARNING @ 13:31:22] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 13:31:22] [setup] RAM Tracking...
[codecarbon INFO @ 13:31:22] [setup] CPU Tracking...


Epoch 100/100 complete.
--- Training Finished ---


[codecarbon WARNING @ 13:31:23] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 13:31:23] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 13:31:23] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.00GHz
[codecarbon WARNING @ 13:31:23] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 13:31:23] [setup] GPU Tracking...
[codecarbon INFO @ 13:31:23] Tracking Nvidia GPU via pynvml
[codecarbon WARNING @ 13:31:23] Failed to retrieve gpu total energy consumption
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/codecarbon/core/gpu.py", line 116, in _get_total_energy_consumption
    return pynvml.nvmlDeviceGetTotalEnergyConsumption(self.handle)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


--- Performance Metrics (ResNet-18 on CIFAR-10) ---
Accuracy: 92.18%
Precision (Macro): 0.9222
Recall (Macro): 0.9218
F1-Score (Macro): 0.9216

--- Green AI / Efficiency Metrics (ResNet-18 on CIFAR-10) ---
Parameter Count: 11.174 M
Model Size: 42.702 MB
Computational Complexity: 0.5579 GFLOPs
Average Inference Latency: 0.0297 ms/image

Energy/CO2 metrics for the evaluation phase tracked by CodeCarbon.
Check the 'emissions.csv' file for details.


# **EcoConvNet**

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time
import os
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from thop import profile
from codecarbon import track_emissions

# --- Configuration ---
BATCH_SIZE = 128
EPOCHS = 100
LEARNING_RATE = 0.001
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"--- EcoConvNet (Proposed Model) on CIFAR-10 ---")
print(f"Using device: {DEVICE}")

# --- Data Loading & Transformation for CIFAR-10 ---
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


class WeightedPooling(nn.Module):
    def __init__(self, input_dim: int, alpha: float = 1.0):
        super().__init__()
        self.attention_scorer = nn.Sequential(
            nn.Linear(input_dim, input_dim // 2), nn.Tanh(), nn.Linear(input_dim // 2, 1)
        )
        self.alpha = alpha

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        B, T, D = h.shape
        device = h.device
        content_scores = self.attention_scorer(h).squeeze(-1)
        pos_indices = torch.arange(T, device=device, dtype=torch.float32)
        normalized_pos = pos_indices / (T - 1) if T > 1 else torch.zeros_like(pos_indices)
        positional_bias = self.alpha * normalized_pos
        combined_scores = content_scores + positional_bias
        weights = torch.softmax(combined_scores, dim=1)
        pooled_vector = torch.sum(h * weights.unsqueeze(-1), dim=1)
        return pooled_vector

class EcoConvNet(nn.Module):
    # Default parameters are now set for CIFAR-10
    def __init__(self, img_size=(32, 32), patch_size=(1, 1), in_channels=3, cnn_out_channels=32,
                 embed_dim=64, num_classes=10, dropout=0.3, pos_bias_alpha=1.0):
        super().__init__()
        self.patch_size = patch_size
        self.cnn_backbone = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, 1, 1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, cnn_out_channels, 3, 1, 1), nn.BatchNorm2d(cnn_out_channels), nn.ReLU(), nn.MaxPool2d(2, 2),
        )
        # Auto-calculate feature map size
        with torch.no_grad():
            dummy_input = torch.zeros(1, in_channels, *img_size)
            feature_map = self.cnn_backbone(dummy_input)
            _, C, H_f, W_f = feature_map.shape
        
        ph, pw = self.patch_size
        self.num_patches = (H_f // ph) * (W_f // pw)
        patch_dim = C * ph * pw
        self.patch_projection = nn.Linear(patch_dim, embed_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, self.num_patches, embed_dim))
        
        self.sequence_processor = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1), nn.BatchNorm1d(embed_dim), nn.ReLU(),
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1), nn.BatchNorm1d(embed_dim), nn.ReLU(),
        )
        self.pooling = WeightedPooling(input_dim=embed_dim, alpha=pos_bias_alpha)
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f = self.cnn_backbone(x)
        B, C, H_f, W_f = f.shape
        ph, pw = self.patch_size
        patches = f.unfold(2, ph, ph).unfold(3, pw, pw).permute(0, 2, 3, 1, 4, 5).contiguous()
        patches = patches.view(B, self.num_patches, -1)
        embeddings = self.patch_projection(patches) + self.positional_encoding
        embeddings_permuted = embeddings.permute(0, 2, 1)
        seq_out = self.sequence_processor(embeddings_permuted)
        seq_out_permuted = seq_out.permute(0, 2, 1)
        pooled_out = self.pooling(seq_out_permuted)
        logits = self.classifier(pooled_out)
        return logits

model = EcoConvNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

def train_model(model, device, train_loader, optimizer, criterion, epochs):
    model.train()
    print("--- Starting Training ---")
    for epoch in range(epochs):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad(); output = model(data); loss = criterion(output, target)
            loss.backward(); optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} complete.")
    print("--- Training Finished ---")

@track_emissions(project_name="EcoConvNet_CIFAR10_Evaluation")
def evaluate_model(model, device, test_loader):
    model.eval()
    all_preds, all_targets, total_latency, num_samples = [], [], 0, 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            start_time = time.perf_counter()
            output = model(data)
            total_latency += (time.perf_counter() - start_time)
            num_samples += data.size(0)
            _, predicted = torch.max(output.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
    return all_targets, all_preds, (total_latency / num_samples) * 1000


# --- Train and Evaluate ---
train_model(model, DEVICE, train_loader, optimizer, criterion, epochs=EPOCHS)
true_labels, predicted_labels, avg_latency_ms = evaluate_model(model, DEVICE, test_loader)

# --- Performance Metrics ---
print("\n--- Performance Metrics (EcoConvNet on CIFAR-10) ---")
accuracy = accuracy_score(true_labels, predicted_labels)
precision, recall, f1_score, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='macro', zero_division=0)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Precision (Macro): {precision:.4f}")
print(f"Recall (Macro): {recall:.4f}")
print(f"F1-Score (Macro): {f1_score:.4f}")

# --- Green AI / Efficiency Metrics ---
print("\n--- Green AI / Efficiency Metrics (EcoConvNet on CIFAR-10) ---")

# 1. Parameter Count & Model Size
# CORRECTED LINE: Changed .num_el() to .numel()
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameter Count: {param_count / 1e6:.3f} M")
torch.save(model.state_dict(), "ecoconvnet_cifar10.pth")
model_size_mb = os.path.getsize("ecoconvnet_cifar10.pth") / (1024 * 1024)
os.remove("ecoconvnet_cifar10.pth")
print(f"Model Size: {model_size_mb:.3f} MB")

# 2. Computational Complexity (FLOPs)
dummy_input = torch.randn(1, 3, 32, 32).to(DEVICE)
flops, _ = profile(model, inputs=(dummy_input,), verbose=False)
print(f"Computational Complexity: {flops / 1e9:.4f} GFLOPs")

# 3. Inference Latency
print(f"Average Inference Latency: {avg_latency_ms:.4f} ms/image")

# 4. Energy Consumption
print("\nEnergy/CO2 metrics for the evaluation phase tracked by CodeCarbon.")
print("Check the 'emissions.csv' file for details.")

--- EcoConvNet (Proposed Model) on CIFAR-10 ---
Using device: cuda
--- Starting Training ---
Epoch 10/100 complete.
Epoch 20/100 complete.
Epoch 30/100 complete.
Epoch 40/100 complete.
Epoch 50/100 complete.
Epoch 60/100 complete.
Epoch 70/100 complete.
Epoch 80/100 complete.
Epoch 90/100 complete.


[codecarbon WARNING @ 14:51:25] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 14:51:25] [setup] RAM Tracking...
[codecarbon INFO @ 14:51:25] [setup] CPU Tracking...


Epoch 100/100 complete.
--- Training Finished ---


[codecarbon WARNING @ 14:51:26] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please contact us.
[codecarbon WARNING @ 14:51:26] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 14:51:26] CPU Model on constant consumption mode: Intel(R) Xeon(R) CPU @ 2.00GHz
[codecarbon WARNING @ 14:51:26] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 14:51:26] [setup] GPU Tracking...
[codecarbon INFO @ 14:51:26] Tracking Nvidia GPU via pynvml
[codecarbon WARNING @ 14:51:26] Failed to retrieve gpu total energy consumption
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/codecarbon/core/gpu.py", line 116, in _get_total_energy_consumption
    return pynvml.nvmlDeviceGetTotalEnergyConsumption(self.handle)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


--- Performance Metrics (EcoConvNet on CIFAR-10) ---
Accuracy: 82.20%
Precision (Macro): 0.8248
Recall (Macro): 0.8220
F1-Score (Macro): 0.8220

--- Green AI / Efficiency Metrics (EcoConvNet on CIFAR-10) ---
Parameter Count: 0.048 M
Model Size: 0.199 MB
Computational Complexity: 0.0036 GFLOPs
Average Inference Latency: 0.0175 ms/image

Energy/CO2 metrics for the evaluation phase tracked by CodeCarbon.
Check the 'emissions.csv' file for details.


# **Ablation Study EcoConvNet**

In [1]:
# Install required libraries for profiling and evaluation
!pip install thop -q
!pip install codecarbon -q
!pip install scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 31.9 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 85.7 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time
import os
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from thop import profile
from codecarbon import EmissionsTracker

# --- Configuration ---
BATCH_SIZE = 128
EPOCHS = 100
LEARNING_RATE = 0.001
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"--- Ablation Study on CIFAR-10 ---")
print(f"Using device: {DEVICE}")

# --- Data Loading & Transformation for CIFAR-10 ---
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ============================================================================
# MODEL DEFINITIONS
# ============================================================================

# --- Component for the full model ---
class WeightedPooling(nn.Module):
    def __init__(self, input_dim: int, alpha: float = 1.0):
        super().__init__()
        self.attention_scorer = nn.Sequential(nn.Linear(input_dim, input_dim // 2), nn.Tanh(), nn.Linear(input_dim // 2, 1))
        self.alpha = alpha
    def forward(self, h: torch.Tensor) -> torch.Tensor:
        B, T, D = h.shape; device = h.device
        content_scores = self.attention_scorer(h).squeeze(-1)
        pos_indices = torch.arange(T, device=device, dtype=torch.float32)
        normalized_pos = pos_indices / (T - 1) if T > 1 else torch.zeros_like(pos_indices)
        positional_bias = self.alpha * normalized_pos
        combined_scores = content_scores + positional_bias
        weights = torch.softmax(combined_scores, dim=1)
        return torch.sum(h * weights.unsqueeze(-1), dim=1)

# --- 1. Full EcoConvNet Model ---
class EcoConvNet(nn.Module):
    def __init__(self, img_size=(32, 32), in_channels=3, cnn_out_channels=32, embed_dim=64, num_classes=10, dropout=0.3):
        super().__init__()
        self.cnn_backbone = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, 1, 1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, cnn_out_channels, 3, 1, 1), nn.BatchNorm2d(cnn_out_channels), nn.ReLU(), nn.MaxPool2d(2, 2),
        )
        with torch.no_grad(): _, C, H_f, W_f = self.cnn_backbone(torch.zeros(1, in_channels, *img_size)).shape
        self.num_patches = H_f * W_f
        self.patch_projection = nn.Linear(C, embed_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, self.num_patches, embed_dim))
        self.sequence_processor = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1), nn.BatchNorm1d(embed_dim), nn.ReLU(),
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1), nn.BatchNorm1d(embed_dim), nn.ReLU(),
        )
        self.pooling = WeightedPooling(input_dim=embed_dim)
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, num_classes)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f = self.cnn_backbone(x)
        B, C, H_f, W_f = f.shape
        patches = f.permute(0, 2, 3, 1).contiguous().view(B, self.num_patches, C)
        embeddings = self.patch_projection(patches) + self.positional_encoding
        seq_out = self.sequence_processor(embeddings.permute(0, 2, 1)).permute(0, 2, 1)
        pooled_out = self.pooling(seq_out)
        return self.classifier(pooled_out)

# --- 2. Ablation Model (No Sequence Processor) ---
class EcoConvNet_NoProcessor(nn.Module):
    def __init__(self, img_size=(32, 32), in_channels=3, cnn_out_channels=32, embed_dim=64, num_classes=10, dropout=0.3):
        super().__init__()
        self.cnn_backbone = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, 1, 1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, cnn_out_channels, 3, 1, 1), nn.BatchNorm2d(cnn_out_channels), nn.ReLU(), nn.MaxPool2d(2, 2),
        )
        with torch.no_grad(): _, C, H_f, W_f = self.cnn_backbone(torch.zeros(1, in_channels, *img_size)).shape
        self.num_patches = H_f * W_f
        self.patch_projection = nn.Linear(C, embed_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, self.num_patches, embed_dim))
        self.pooling = WeightedPooling(input_dim=embed_dim)
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, num_classes)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f = self.cnn_backbone(x)
        B, C, H_f, W_f = f.shape
        patches = f.permute(0, 2, 3, 1).contiguous().view(B, self.num_patches, C)
        embeddings = self.patch_projection(patches) + self.positional_encoding
        pooled_out = self.pooling(embeddings) # Skip sequence processor
        return self.classifier(pooled_out)

# --- 3. Ablation Model (Global Average Pooling) ---
class EcoConvNet_AvgPool(nn.Module):
    def __init__(self, img_size=(32, 32), in_channels=3, cnn_out_channels=32, embed_dim=64, num_classes=10, dropout=0.3):
        super().__init__()
        self.cnn_backbone = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, 1, 1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, cnn_out_channels, 3, 1, 1), nn.BatchNorm2d(cnn_out_channels), nn.ReLU(), nn.MaxPool2d(2, 2),
        )
        with torch.no_grad(): _, C, H_f, W_f = self.cnn_backbone(torch.zeros(1, in_channels, *img_size)).shape
        self.num_patches = H_f * W_f
        self.patch_projection = nn.Linear(C, embed_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, self.num_patches, embed_dim))
        self.sequence_processor = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1), nn.BatchNorm1d(embed_dim), nn.ReLU(),
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1), nn.BatchNorm1d(embed_dim), nn.ReLU(),
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim), nn.Linear(embed_dim, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, num_classes)
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f = self.cnn_backbone(x)
        B, C, H_f, W_f = f.shape
        patches = f.permute(0, 2, 3, 1).contiguous().view(B, self.num_patches, C)
        embeddings = self.patch_projection(patches) + self.positional_encoding
        seq_out = self.sequence_processor(embeddings.permute(0, 2, 1)).permute(0, 2, 1)
        pooled_out = torch.mean(seq_out, dim=1) # Use simple average pooling
        return self.classifier(pooled_out)

--- Ablation Study on CIFAR-10 ---
Using device: cuda


100%|██████████| 170M/170M [00:05<00:00, 29.5MB/s] 


In [6]:
import logging

# ============================================================================
# CORRECTED GENERIC TRAINING AND ANALYSIS FUNCTIONS
# ============================================================================

def train_model(model, device, train_loader, optimizer, criterion, epochs):
    model.train()
    print(f"--- Starting Training for {epochs} epochs ---")
    for epoch in range(epochs):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad(); output = model(data); loss = criterion(output, target)
            loss.backward(); optimizer.step()
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{epochs} complete.")
    print("--- Training Finished ---")

def evaluate_and_analyze(model, model_name, device, test_loader):
    # FIX 2: Set a higher log level for codecarbon to reduce verbose warnings
    # This will hide the NVMLError tracebacks which are expected in this environment
    codecarbon_logger = logging.getLogger("codecarbon")
    codecarbon_logger.setLevel(logging.ERROR)
    
    tracker = EmissionsTracker(project_name=model_name, log_level='error')
    tracker.start()
    
    model.eval()
    all_preds, all_targets, total_latency, num_samples = [], [], 0, 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            start_time = time.perf_counter()
            output = model(data)
            total_latency += (time.perf_counter() - start_time)
            num_samples += data.size(0)
            _, predicted = torch.max(output.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(target.cpu().numpy())
            
    emissions_data = tracker.stop()
    avg_latency_ms = (total_latency / num_samples) * 1000

    print(f"\n--- Performance Metrics ({model_name}) ---")
    accuracy = accuracy_score(all_targets, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='macro', zero_division=0)
    print(f"Accuracy: {accuracy * 100:.2f}%\nPrecision: {precision:.4f}\nRecall: {recall:.4f}\nF1-Score: {f1:.4f}")

    print(f"\n--- Green AI / Efficiency Metrics ({model_name}) ---")
    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameter Count: {param_count / 1e6:.3f} M")
    torch.save(model.state_dict(), "temp_model.pth"); model_size_mb = os.path.getsize("temp_model.pth")/(1024*1024); os.remove("temp_model.pth")
    print(f"Model Size: {model_size_mb:.3f} MB")
    
    dummy_input = torch.randn(1, 3, 32, 32).to(device)
    flops, _ = profile(model, inputs=(dummy_input,), verbose=False)
    print(f"Computational Complexity: {flops / 1e9:.4f} GFLOPs")
    print(f"Average Inference Latency: {avg_latency_ms:.4f} ms/image")

    # FIX 1: Handle both object and float return types from tracker.stop()
    if hasattr(emissions_data, 'energy_consumed'):
        # This is the expected case where a full EmissionsData object is returned
        energy_kwh = emissions_data.energy_consumed
        emissions_kg = emissions_data.emissions
        print(f"Energy Consumed (Evaluation): {energy_kwh * 1e6:.2f} x10^-6 kWh")
        print(f"CO2 Emissions (Evaluation): {emissions_kg * 1e6:.2f} x10^-6 kg")
    elif isinstance(emissions_data, float):
        # This is the fallback case where only the CO2 emissions float is returned
        emissions_kg = emissions_data
        print("Energy Consumed (Evaluation): Not available directly (CodeCarbon fallback).")
        print(f"CO2 Emissions (Evaluation): {emissions_kg * 1e6:.2f} x10^-6 kg")
    else:
        print("Could not parse energy consumption data.")
        
# ============================================================================
# MAIN EXECUTION LOOP (Starts all experiments)
# ============================================================================
models_to_evaluate = [
    ("EcoConvNet (Full Model)", EcoConvNet),
    ("EcoConvNet (w/o Sequence Processor)", EcoConvNet_NoProcessor),
    ("EcoConvNet (w/ Avg Pooling)", EcoConvNet_AvgPool)
]

for model_name, model_class in models_to_evaluate:
    print("\n" + "="*80)
    print(f"NOW EVALUATING: {model_name.upper()}")
    print("="*80)

    model = model_class().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    train_model(model, DEVICE, train_loader, optimizer, criterion, epochs=EPOCHS)
    evaluate_and_analyze(model, model_name, DEVICE, test_loader)

print("\n\n--- Ablation Study Complete ---")


NOW EVALUATING: ECOCONVNET (FULL MODEL)
--- Starting Training for 100 epochs ---
Epoch 20/100 complete.
Epoch 40/100 complete.
Epoch 60/100 complete.
Epoch 80/100 complete.
Epoch 100/100 complete.
--- Training Finished ---

--- Performance Metrics (EcoConvNet (Full Model)) ---
Accuracy: 81.55%
Precision: 0.8155
Recall: 0.8155
F1-Score: 0.8144

--- Green AI / Efficiency Metrics (EcoConvNet (Full Model)) ---
Parameter Count: 0.048 M
Model Size: 0.199 MB
Computational Complexity: 0.0036 GFLOPs
Average Inference Latency: 0.0162 ms/image
Energy Consumed (Evaluation): Not available directly (CodeCarbon fallback).
CO2 Emissions (Evaluation): 6.60 x10^-6 kg

NOW EVALUATING: ECOCONVNET (W/O SEQUENCE PROCESSOR)
--- Starting Training for 100 epochs ---
Epoch 20/100 complete.
Epoch 40/100 complete.
Epoch 60/100 complete.
Epoch 80/100 complete.
Epoch 100/100 complete.
--- Training Finished ---

--- Performance Metrics (EcoConvNet (w/o Sequence Processor)) ---
Accuracy: 70.76%
Precision: 0.7133
Rec